# Assignment 2 Part 3: Background Subtraction in Summarized Video

## Objectives:
1. **Tune parameters** of background subtractor for the summarized video (40 marks)
2. **Compare foreground/background** of:
   - (i) Without summarization (full video)
   - (ii) With summarization (keyframes only)
   (40 marks)

---

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print("="*80)
print("PART 3: BACKGROUND SUBTRACTION - SUMMARIZED VS FULL VIDEO COMPARISON")
print("="*80)

## STEP 1: Load Videos
Load both the original full video and the pre-created summarized video

In [ ]:
print("\n" + "="*80)
print("STEP 1: LOAD VIDEOS")
print("="*80)

# File paths
full_video_path = r'C:\Users\Jyotirmoy Nath\Downloads\Assignment1\Assignment1\umcp.mpg'
summary_video_path = r'C:\Users\Jyotirmoy Nath\Downloads\Assignment1\Assignment1\video_summary.MP4'

# Load FULL video
print("\nLoading FULL video...")
cap_full = cv2.VideoCapture(full_video_path)
if not cap_full.isOpened():
    raise FileNotFoundError(f"Cannot open video: {full_video_path}")

fps_full = cap_full.get(cv2.CAP_PROP_FPS)
width = int(cap_full.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap_full.get(cv2.CAP_PROP_FRAME_HEIGHT))

full_frames = []
while True:
    ret, frame = cap_full.read()
    if not ret:
        break
    full_frames.append(frame)
cap_full.release()

print(f"✓ Full video loaded:")
print(f"  Frames: {len(full_frames)}")
print(f"  Size: {width}×{height}")
print(f"  FPS: {fps_full:.2f}")
print(f"  Duration: {len(full_frames)/fps_full:.2f} seconds")

# Load SUMMARIZED video
print("\nLoading SUMMARIZED video...")
cap_summary = cv2.VideoCapture(summary_video_path)
if not cap_summary.isOpened():
    raise FileNotFoundError(f"Cannot open video: {summary_video_path}")

fps_summary = cap_summary.get(cv2.CAP_PROP_FPS)
width_summary = int(cap_summary.get(cv2.CAP_PROP_FRAME_WIDTH))
height_summary = int(cap_summary.get(cv2.CAP_PROP_FRAME_HEIGHT))

summary_frames = []
while True:
    ret, frame = cap_summary.read()
    if not ret:
        break
    summary_frames.append(frame)
cap_summary.release()

print(f"✓ Summarized video loaded:")
print(f"  Frames: {len(summary_frames)}")
print(f"  Size: {width_summary}×{height_summary}")
print(f"  FPS: {fps_summary:.2f}")
print(f"  Duration: {len(summary_frames)/fps_summary:.2f} seconds")

print(f"\n✓ Compression ratio: {len(full_frames)/len(summary_frames):.1f}x")
print(f"✓ Time reduction: {(1 - len(summary_frames)/len(full_frames))*100:.1f}%")

## STEP 2: Background Subtraction Pipeline

Define the complete background subtraction function using Sequential Labeling method from Part 2

In [ ]:
print("\n" + "="*80)
print("STEP 2: DEFINE BACKGROUND SUBTRACTION FUNCTION")
print("="*80)

def background_subtraction_pipeline(frames, threshold=25, min_area=50, 
                                   morph_close_iter=2, morph_open_iter=1,
                                   resize_width=160, resize_height=120):
    """
    Complete background subtraction pipeline using Sequential Labeling method
    
    Parameters:
    -----------
    frames : list
        List of video frames (BGR images)
    threshold : int (15-50)
        Motion detection threshold. Higher = less sensitive
    min_area : int (25-200)
        Minimum blob area in pixels to keep. Higher = more filtering
    morph_close_iter : int (1-3)
        Morphological closing iterations. Fills holes in motion
    morph_open_iter : int (1-3)
        Morphological opening iterations. Removes noise
    resize_width, resize_height : int
        Processing resolution for faster computation
    
    Returns:
    --------
    foreground_frames : list
        Frames with only moving objects (background = black)
    background_frames : list
        Frames with only static scene (foreground = black)
    masks : list
        Binary masks (255 = motion, 0 = static)
    """
    
    print(f"\nBackground Subtraction Configuration:")
    print(f"  Threshold (τ): {threshold}")
    print(f"  Min area: {min_area} pixels")
    print(f"  Morph close iterations: {morph_close_iter}")
    print(f"  Morph open iterations: {morph_open_iter}")
    print(f"  Processing resolution: {resize_width}×{resize_height}")
    print(f"  Input frames: {len(frames)}")
    
    # STEP 2.1: PREPROCESSING
    print(f"\n  [1/4] Preprocessing...")
    preprocessed = []
    for i, frame in enumerate(frames):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (resize_width, resize_height))
        preprocessed.append(resized.astype(np.float32))
        
        if (i+1) % 100 == 0:
            print(f"        Preprocessed {i+1}/{len(frames)} frames")
    
    print(f"        ✓ Preprocessed: {len(preprocessed)} frames")
    
    # STEP 2.2: MOTION ESTIMATION
    print(f"\n  [2/4] Motion estimation (frame differencing)...")
    motion_images = []
    for t in range(1, len(preprocessed)):
        # D_t = |F_t - F_{t-1}|
        diff = np.abs(preprocessed[t] - preprocessed[t-1])
        motion_images.append(diff)
        
        if t % 100 == 0:
            print(f"        Computed motion for frame {t}/{len(preprocessed)}")
    
    print(f"        ✓ Motion images: {len(motion_images)}")
    
    # STEP 2.3: SEQUENTIAL LABELING SEGMENTATION
    print(f"\n  [3/4] Sequential labeling segmentation...")
    masks = []
    kernel = np.ones((3, 3), np.uint8)
    
    for i, motion in enumerate(motion_images):
        # Threshold: M_t(i,j) = 1 if D_t(i,j) > τ, else 0
        binary_mask = (motion > threshold).astype(np.uint8) * 255
        
        # Morphological closing: fill small holes
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel, 
                                       iterations=morph_close_iter)
        
        # Morphological opening: remove small noise
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel, 
                                       iterations=morph_open_iter)
        
        # Connected components analysis
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
            binary_mask, connectivity=8
        )
        
        # Filter by minimum area
        final_mask = np.zeros_like(binary_mask)
        for label in range(1, num_labels):
            area = stats[label, cv2.CC_STAT_AREA]
            if area >= min_area:
                final_mask[labels == label] = 255
        
        # Resize mask back to original resolution
        h, w = frames[0].shape[:2]
        final_mask = cv2.resize(final_mask, (w, h))
        masks.append(final_mask)
        
        if (i+1) % 100 == 0:
            print(f"        Segmented {i+1}/{len(motion_images)} frames")
    
    print(f"        ✓ Masks generated: {len(masks)}")
    
    # STEP 2.4: EXTRACT FOREGROUND AND BACKGROUND
    print(f"\n  [4/4] Extracting foreground and background...")
    foreground_frames = []
    background_frames = []
    
    # First frame: no motion data available
    foreground_frames.append(np.zeros_like(frames[0]))
    background_frames.append(frames[0].copy())
    
    # Process remaining frames
    for i in range(len(masks)):
        frame = frames[i+1]
        mask = masks[i]
        
        # Foreground: F^fg_t = F_t ⊙ M_t
        mask_3ch = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        fg = cv2.bitwise_and(frame, mask_3ch)
        foreground_frames.append(fg)
        
        # Background: F^bg_t = F_t ⊙ (1 - M_t)
        mask_inv = cv2.bitwise_not(mask)
        mask_inv_3ch = cv2.cvtColor(mask_inv, cv2.COLOR_GRAY2BGR)
        bg = cv2.bitwise_and(frame, mask_inv_3ch)
        background_frames.append(bg)
    
    print(f"        ✓ Foreground frames: {len(foreground_frames)}")
    print(f"        ✓ Background frames: {len(background_frames)}")
    
    print(f"\n✓ Background subtraction pipeline complete!")
    
    return foreground_frames, background_frames, masks

print("\n✓ Background subtraction function defined")

## STEP 3: Apply Background Subtraction to FULL Video

### ⚙️ Parameters for Full Video (Baseline)

In [ ]:
print("\n" + "="*80)
print("STEP 3: BACKGROUND SUBTRACTION ON FULL VIDEO (WITHOUT SUMMARIZATION)")
print("="*80)

# ============================================================================
# FULL VIDEO PARAMETERS - BASELINE SETTINGS
# ============================================================================
# These are optimized for consecutive frames (small temporal gaps)
FULL_THRESHOLD = 25        # Motion threshold
FULL_MIN_AREA = 50         # Minimum blob area (pixels)
FULL_MORPH_CLOSE = 2       # Morphological close iterations
FULL_MORPH_OPEN = 1        # Morphological open iterations
# ============================================================================

print("\nApplying background subtraction to FULL video...")
print(f"Parameters:")
print(f"  Threshold: {FULL_THRESHOLD}")
print(f"  Min area: {FULL_MIN_AREA}")
print(f"  Morph close: {FULL_MORPH_CLOSE}")
print(f"  Morph open: {FULL_MORPH_OPEN}")

full_fg, full_bg, full_masks = background_subtraction_pipeline(
    full_frames,
    threshold=FULL_THRESHOLD,
    min_area=FULL_MIN_AREA,
    morph_close_iter=FULL_MORPH_CLOSE,
    morph_open_iter=FULL_MORPH_OPEN
)

print(f"\n" + "="*80)
print("FULL VIDEO PROCESSING COMPLETE")
print("="*80)

## STEP 4: Apply Background Subtraction to SUMMARIZED Video

### ⚙️ TUNED Parameters for Summarized Video

**KEY INSIGHT:** Summarized video has **LARGER TEMPORAL GAPS** between frames.

**Solution:** Adjust parameters to account for larger motion differences

In [ ]:
print("\n" + "="*80)
print("STEP 4: BACKGROUND SUBTRACTION ON SUMMARIZED VIDEO (WITH SUMMARIZATION)")
print("="*80)

# ============================================================================
# SUMMARIZED VIDEO PARAMETERS - TUNED FOR LARGER TEMPORAL GAPS
# ============================================================================
# ADJUST THESE VALUES TO OPTIMIZE FOR SUMMARIZED VIDEO
# ============================================================================

SUMMARY_THRESHOLD = 35     # ← HIGHER than full video (try 30-45)
SUMMARY_MIN_AREA = 100     # ← HIGHER than full video (try 75-150)
SUMMARY_MORPH_CLOSE = 3    # ← MORE iterations (try 2-4)
SUMMARY_MORPH_OPEN = 2     # ← MORE iterations (try 1-3)

# ============================================================================

print("\nTuned parameters for summarized video:")
print(f"  Threshold: {SUMMARY_THRESHOLD} (vs {FULL_THRESHOLD} for full)")
print(f"  Min area: {SUMMARY_MIN_AREA} (vs {FULL_MIN_AREA} for full)")
print(f"  Morph close: {SUMMARY_MORPH_CLOSE} (vs {FULL_MORPH_CLOSE} for full)")
print(f"  Morph open: {SUMMARY_MORPH_OPEN} (vs {FULL_MORPH_OPEN} for full)")

print("\nApplying background subtraction to SUMMARIZED video...")

summary_fg, summary_bg, summary_masks = background_subtraction_pipeline(
    summary_frames,
    threshold=SUMMARY_THRESHOLD,
    min_area=SUMMARY_MIN_AREA,
    morph_close_iter=SUMMARY_MORPH_CLOSE,
    morph_open_iter=SUMMARY_MORPH_OPEN
)

print(f"\n" + "="*80)
print("SUMMARIZED VIDEO PROCESSING COMPLETE")
print("="*80)

## STEP 5: Visual Comparison

In [ ]:
print("\n" + "="*80)
print("STEP 5: VISUAL COMPARISON")
print("="*80)

num_samples = min(8, len(summary_frames))
sample_indices = np.linspace(1, len(summary_frames)-1, num_samples, dtype=int)
full_sample_indices = np.linspace(1, len(full_frames)-1, num_samples, dtype=int)

fig, axes = plt.subplots(num_samples, 4, figsize=(20, 3*num_samples))
fig.suptitle('COMPARISON: Full Video vs Summarized Video', 
             fontsize=18, fontweight='bold', y=0.995)

for i, (full_idx, summary_idx) in enumerate(zip(full_sample_indices, sample_indices)):
    axes[i, 0].imshow(cv2.cvtColor(full_fg[full_idx], cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f'Full Video\nForeground\nFrame {full_idx}', fontsize=10)
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(cv2.cvtColor(full_bg[full_idx], cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(f'Full Video\nBackground\nFrame {full_idx}', fontsize=10)
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(cv2.cvtColor(summary_fg[summary_idx], cv2.COLOR_BGR2RGB))
    axes[i, 2].set_title(f'Summary\nForeground\nFrame {summary_idx}', fontsize=10)
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(cv2.cvtColor(summary_bg[summary_idx], cv2.COLOR_BGR2RGB))
    axes[i, 3].set_title(f'Summary\nBackground\nFrame {summary_idx}', fontsize=10)
    axes[i, 3].axis('off')

plt.tight_layout()
plt.savefig('comparison_full_vs_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visual comparison saved: comparison_full_vs_summary.png")

## STEP 6: Quantitative Comparison

In [ ]:
print("\n" + "="*80)
print("STEP 6: QUANTITATIVE COMPARISON")
print("="*80)

def compute_metrics(masks):
    fg_ratios = []
    for mask in masks:
        if mask is not None:
            fg_pixels = np.sum(mask > 0)
            total_pixels = mask.shape[0] * mask.shape[1]
            fg_ratios.append(fg_pixels / total_pixels)
    
    num_components = []
    for mask in masks:
        if mask is not None:
            num_labels, _ = cv2.connectedComponents(mask)
            num_components.append(num_labels - 1)
    
    return {
        'avg_fg_ratio': np.mean(fg_ratios) if fg_ratios else 0,
        'std_fg_ratio': np.std(fg_ratios) if fg_ratios else 0,
        'consistency': 1.0 / (1.0 + np.std(fg_ratios)) if fg_ratios else 1.0,
        'avg_components': np.mean(num_components) if num_components else 0
    }

full_metrics = compute_metrics(full_masks)
summary_metrics = compute_metrics(summary_masks)

print(f"\n{'Metric':<30} {'Full Video':<20} {'Summarized':<20}")
print("-" * 70)
print(f"{'Frames':<30} {len(full_frames):<20} {len(summary_frames):<20}")
print(f"{'Compression':<30} {'1.0x':<20} {f'{len(full_frames)/len(summary_frames):.1f}x':<20}")
print(f"{'Avg FG ratio':<30} {full_metrics['avg_fg_ratio']:<20.4f} {summary_metrics['avg_fg_ratio']:<20.4f}")
print(f"{'Consistency':<30} {full_metrics['consistency']:<20.4f} {summary_metrics['consistency']:<20.4f}")
print(f"{'Avg components':<30} {full_metrics['avg_components']:<20.2f} {summary_metrics['avg_components']:<20.2f}")
print()
print(f"{'Parameter':<30} {'Full':<20} {'Summary':<20}")
print("-" * 70)
print(f"{'Threshold':<30} {FULL_THRESHOLD:<20} {SUMMARY_THRESHOLD:<20}")
print(f"{'Min area':<30} {FULL_MIN_AREA:<20} {SUMMARY_MIN_AREA:<20}")
print(f"{'Morph close':<30} {FULL_MORPH_CLOSE:<20} {SUMMARY_MORPH_CLOSE:<20}")
print(f"{'Morph open':<30} {FULL_MORPH_OPEN:<20} {SUMMARY_MORPH_OPEN:<20}")

## STEP 7: Save Output Videos

In [ ]:
print("\n" + "="*80)
print("STEP 7: SAVE OUTPUT VIDEOS")
print("="*80)

def save_video(frames, filename, fps_rate):
    if len(frames) == 0:
        return
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(filename, fourcc, fps_rate, (w, h))
    for frame in frames:
        out.write(frame)
    out.release()
    print(f"  ✓ {filename}")

print("\nSaving outputs...")
save_video(full_fg, 'full_foreground.mp4', fps_full)
save_video(full_bg, 'full_background.mp4', fps_full)
save_video(summary_fg, 'summary_foreground.mp4', fps_summary)
save_video(summary_bg, 'summary_background.mp4', fps_summary)
print("\n✓ All videos saved!")